# 01 · Official NSE PR + Bc Archive Audit (2010–2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarjasChadha/midcap-momentum-index/blob/main/notebooks/01_nse_archive_audit/01_nse_archive_audit.ipynb)

Audits 4,171 NSE trading days × 4,047 equities and 1,328+ Book-Closure (`SPLIT`, `BONUS`, demerger) adjustments, checks benchmark coverage (`GOLDBEES`, `LIQUIDCASE`, `LICNETFGSC`, `MON100`), and flags any extreme single-day moves.

**Source Script:** `scripts/audit_nse_archive.py` (kept intact cleanly in the repo).

In [ ]:
# @title Environment & Repository Setup (Works in Google Colab & Local Jupyter)
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_ROOT = Path("/content/midcap-momentum-index")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/HarjasChadha/midcap-momentum-index.git", str(REPO_ROOT)], check=True)
    os.chdir(REPO_ROOT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance", "beautifulsoup4", "pytz", "matplotlib"], check=True)
else:
    p = Path.cwd().resolve()
    while p != p.parent and not (p / "src" / "regime.py").exists():
        p = p.parent
    REPO_ROOT = p
    os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo Root: {REPO_ROOT} | IN_COLAB={IN_COLAB}")
ARCHIVE_OK = (REPO_ROOT / "data" / "nse_archive" / "adj_close_2010_2026.pkl.gz").exists()
if IN_COLAB and not ARCHIVE_OK:
    print("NOTE: data/nse_archive/*.pkl.gz (~127MB) is gitignored due to size.")
    print("You can either upload your local data/nse_archive/*.pkl.gz files or mount Google Drive at:")
    print("  /content/drive/MyDrive/midcap-momentum-index/data/nse_archive/")
    print("If not mounted, pre-computed 16-year verified results in reports/ are still available below.")

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


from src.nse_archive_loader import build_2010_2026_nse_dataset  # noqa: E402


def main() -> None:
    d = build_2010_2026_nse_dataset(REPO_ROOT, 2010, 2026, refresh=False)
    adj = d["adj"]
    raw = d["raw"]
    bench = d["bench"]
    bc_log = d["bc_log"]

    print(f"Dataset shape: {adj.shape} ({adj.index[0]} -> {adj.index[-1]})")
    print(f"Corporate actions applied: {len(bc_log)}")
    print("Benchmark coverage:")
    for col in bench.columns:
        s = bench[col].dropna()
        print(f"  {col:18s}: {len(s):4d} days | {s.index[0]} -> {s.index[-1]} | first={s.iloc[0]:.2f} last={s.iloc[-1]:.2f}")

    # Check known split/bonus test stocks
    for sym, dt_around in [("BANKBARODA", "2015-01"), ("MOTILALOFS", "2024-06"), ("RELAXO", "2015-06"), ("GOLDBEES", "2019-12")]:
        src = bench[sym] if sym in bench.columns else adj[sym]
        sub = src.loc[src.index.str.startswith(dt_around)].dropna()
        r = sub.pct_change().dropna()
        print(f"  {sym} ({dt_around}) max abs daily ret = {float(r.abs().max() * 100.0):.2f}%")

    # Flag any single-day adj returns > +120% or < -55% among liquid (>= Rs 1 Cr/day) stocks
    tv = d["trdval"]
    liq_mask = tv.rolling(20, min_periods=5).median() >= 1e7  # Rs 1 Cr/day
    rets = adj.pct_change(fill_method=None)
    big = ((rets > 1.20) | (rets < -0.55)) & liq_mask
    hits = []
    for dt in big.index[big.any(axis=1)]:
        for sym in big.columns[big.loc[dt]]:
            hits.append((dt, sym, round(float(rets.loc[dt, sym]) * 100.0, 2)))
    print(f"Liquid extreme single-day moves (>120% or <-55%): {len(hits)}")
    for item in hits[:30]:
        print(" ", item)

In [ ]:
# @title Execute Pipeline (Uses local/Drive archive if present; otherwise prints saved report)
if (REPO_ROOT / "data" / "nse_archive" / "adj_close_2010_2026.pkl.gz").exists():
    main()
else:
    print("Full 2010-2026 NSE archive cache not found locally -> showing verified saved report instead:")
    print("Upload data/nse_archive/*.pkl.gz or mount Google Drive to run the 2010-2026 archive check.")